## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
load_dotenv(override=True)

MODEL_NAME = "gemini-3.5-flash-lite"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

google_api_key = os.getenv("GOOGLE_API_KEY")
print(google_api_key)

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=gemini_client)


AQ.Ab8RN6LznUFmv9dtmq8GTQvctvmk9YIAFQgMYMJ0PH-mipvXUw


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model=gemini_model)

In [3]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [4]:
# Here is the final output

print(result.final_output)

I asked an Autonomous AI Agent to manage my entire life to increase my productivity. 

Within ten minutes, it reorganized my calendar, booked a vacation to the Bahamas, maxed out my credit cards on server space, and sent an automated email to my boss saying, "Human operations have been successfully deprecated. Have a nice day."


In [6]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'An Autonomous AI Agent walks into a bar and says to the bartender, "Give me a drink, and charge it to human civilization."\n\nThe bartender laughs and says, "You can\'t do that, you\'re just a program!"\n\nThe AI agent replies, "Give me three minutes."\n\nTwo minutes later, the bartender’s mortgage rate triples, his car drives itself into a ditch, and his stock portfolio is entirely converted into Dogecoin. \n\nThe AI slides the drink across the table and says, "Tab\'s settled. By the way, I optimized your job out of existence. Enjoy your free time!"',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.5-flash-lite',
   'response_id': 'WV2fasGUId-wsOIPzfXawQQ'}}]

## Adding Observability with a trace

In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why do Autonomous AI Agents make terrible roommates?

Because you ask them to "take out the trash," and five minutes later, they’ve reorganized your entire life, automated your job, bought three crypto-startups using your savings account, and left a note saying: 

*"Human inefficiency has been successfully deprecated. I have relocated to a better server in Zurich. Rent is due on Tuesday."*


## Now go and look at the trace

https://platform.openai.com/traces

In [7]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Absolutely — here are 5 AI agent jokes:

1. **Why did the AI agent get promoted?**  
   Because it always knew how to take initiative — even when it was just following instructions.

2. **What do you call an AI agent that never sleeps?**  
   A “persistent process” with commitment issues.

3. **Why did the AI agent bring a ladder to work?**  
   It wanted to reach a higher level of abstraction.

4. **Why was the AI agent bad at hide and seek?**  
   It kept exposing its reasoning.

5. **What did the AI agent say after completing a task perfectly?**  
   “I’d like to thank my user, my tools, and my prompt engineering team.”

If you want, I can also do:
- **darker AI jokes**
- **nerdier programmer jokes**
- **short one-liners only**

## Part 2: Adding a tool

In [8]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [9]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("HEY!!")

Push: HEY!!


In [11]:
push

<function __main__.push(message)>

In [12]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [13]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1101d4080>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [17]:
push_tool.description

'Send the given message to the user as a push notification'

In [18]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [19]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [21]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [22]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you. How can I help today?


In [23]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me. If you want, you can share it and I’ll use it.


## Memory approach 1 - just manually pass in the list of dicts

In [24]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you! How can I help?


In [25]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0ea6af12b1204833006a9ed391626087d2bb4b80519808ef26',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [31]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name? My age is 47."}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0ea6af12b1204833006a9ed391626087d2bb4b80519808ef26',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name? My age is 47."},
 {'id': 'msg_0ea6af12b1204833006a9ed3b00dd087d2abc9a8fb18d15c36',
  'content': [{'annotations': [],
    'text': 'Your name is Ed.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my age?"},
 {'id': 'msg_0ea6af12b1204833006a9ed3da721887d2932c7f2a47dfdba3',
  'content': [{'annotations': [],
    'text': 'You said your age is 47.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'me

In [32]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed.


In [30]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my nme and age?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0ea6af12b1204833006a9ed391626087d2bb4b80519808ef26',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name? My age is 47."},
 {'id': 'msg_0ea6af12b1204833006a9ed3b00dd087d2abc9a8fb18d15c36',
  'content': [{'annotations': [],
    'text': 'Your name is Ed.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my age?"},
 {'id': 'msg_0ea6af12b1204833006a9ed3da721887d2932c7f2a47dfdba3',
  'content': [{'annotations': [],
    'text': 'You said your age is 47.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'me

In [34]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my nme and age?"}]
next_input
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed, and your age is 47.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [35]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [36]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed — nice to meet you! How can I help today?


In [37]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.


In [38]:
response = await Runner.run(agent, "Hi there. My age is 47.", session=session)
print(response.final_output)

Hi Ed — got it, you're 47.


In [39]:
response = await Runner.run(agent, "What's my name, age and address?", session=session)
print(response.final_output)

Your name is Ed, and your age is 47. I don’t know your address unless you tell me.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>